# SNPデータ前処理

複数バッチの Genotyping TXT を読み込み、feather で保存する。

1. 各バッチを読み込み・転置し、共通SNPで結合
2. 欠損処理（高欠損の行・列、定数列を除外）
3. 数値変換（AA:0, AB:1, BB:2）・MAF/HWEフィルタ・平均補完
4. メタデータ順にSNP列を並び替え
5. feather 形式で保存（補完版・非補完版）

前処理アルゴリズムの本体は共有パッケージ `jbrt.snp`。

In [7]:
import json
import logging
from datetime import datetime, timedelta, timezone

import pandas as pd

from jbrt import config, snp

# このサーバのTZはUTCなので、日付は日本時間(JST)基準で出す
JST = timezone(timedelta(hours=9))

# jbrt.snp の INFO ログ（各フィルタの除外件数）をノート上に表示する
_logger = logging.getLogger("jbrt")
_logger.setLevel(logging.INFO)
if not _logger.handlers:
    _h = logging.StreamHandler()
    _h.setFormatter(logging.Formatter("%(message)s"))
    _logger.addHandler(_h)
_logger.propagate = False

import re


def fix_snp_id(s):
    """このデータ固有の既知の特例だけを直す（汎用ではない）。"""
    t = s.replace("AX004,AX007", "AX004")     # (1) PJ番号併記 → AX004 に統一
    t = re.sub(r"_\d+\.CEL$", ".CEL", t)      # (2) リラン枝番 _N を除去 (例: _A13_2 → _A13)
    return t

## 設定

In [8]:
RAW_DIR, PROCESSED_DIR = config.RAW_DIR, config.PROCESSED_DIR

# 入力: SNP ディレクトリ配下を再帰的に探し、このパターンに合う TXT を対象にする
INPUT_DIR = RAW_DIR / "SNP"
TXT_PATTERN = "Genotyping_AX*.txt"

# 使うバッチの絞り込み。None なら該当TXTを全て使う。
# 例: SELECT = ["AX003", "AX020"]  # AX番号（ファイル名の部分一致）で選ぶ
SELECT = None

# 入力: アレイ注釈CSV
META_PATH = RAW_DIR / "SNP_meta" / "Axiom_BovMDv3.na35.r2.a2.annot.csv"


# フィルタ閾値
MISSING_THRESHOLD = 0.05   # 欠損率がこの値以上の行・列を除外
MAF_THRESHOLD = 0.01       # MAFがこの値未満のSNPを除外
HWE_THRESHOLD = 1e-4       # HWE検定のp値がこの値未満のSNPを除外

INPUT_DIR

PosixPath('/home/s_mori/JBRT/JBRT_data/raw/SNP')

## 1. 読み込み・結合

各バッチを `サンプル×SNP` に転置し、共通SNP（列の積集合）で縦結合する。

In [9]:
txt_paths = snp.find_genotyping_txt(INPUT_DIR, TXT_PATTERN, SELECT)
print(f"対象TXT: {len(txt_paths)}件" + (f"  SELECT={SELECT}" if SELECT else "  (全件)"))

dfs = []
for p in txt_paths:
    d = snp.load_genotyping_txt(p)
    print(f"{p.name}: {d.shape[0]} サンプル × {d.shape[1]} SNP")
    dfs.append(d)

df = snp.concat_common_snps(dfs)
print(f"結合後: {df.shape[0]} サンプル × {df.shape[1]} SNP")
df.iloc[:3, :5]

対象TXT: 10件  (全件)
Genotyping_AX003.txt: 96 サンプル × 57919 SNP
Genotyping_AX004.txt: 48 サンプル × 56854 SNP
Genotyping_AX020.txt: 669 サンプル × 58995 SNP
Genotyping_AX032.txt: 94 サンプル × 54465 SNP
Genotyping_AX034.txt: 96 サンプル × 58139 SNP
Genotyping_AX037.txt: 95 サンプル × 57864 SNP
Genotyping_AX041.txt: 95 サンプル × 53228 SNP
Genotyping_AX045.txt: 96 サンプル × 56917 SNP
Genotyping_AX050.txt: 96 サンプル × 59078 SNP
Genotyping_AX056.txt: 91 サンプル × 58771 SNP
結合後: 1476 サンプル × 44795 SNP


probeset_id,AX-105094887,AX-106719429,AX-106719435,AX-106719437,AX-106719440
snp_id,,,,,
AX003_A01.CEL,AA,BB,BB,BB,AA
AX003_A03.CEL,AA,BB,BB,BB,AA
AX003_A05.CEL,AA,BB,BB,BB,AA


## 2〜4. 欠損処理・数値変換・並び替え

無効値除去 → 数値変換（AA:0, AB:1, BB:2）→ MAF/HWE フィルタ → 平均補完 → メタデータ順の並び替え 

In [10]:
# ============================================================
# 2. 欠損処理
# ============================================================
print("---- 2. 欠損処理 ----")
df = snp.filter_missing(df, threshold=MISSING_THRESHOLD)
print(f"処理後の形状: {df.shape}")

# ============================================================
# 3. 数値変換・MAF/HWE フィルタ・補完
# ============================================================
print("\n---- 3. 数値変換・MAF/HWE フィルタ・補完 ----")
df_numeric = snp.to_numeric(df)
df_filtered = snp.filter_maf(df_numeric, threshold=MAF_THRESHOLD)
df_filtered = snp.filter_hwe(df_filtered, threshold=HWE_THRESHOLD)

df_not_imputed = df_filtered.copy()
df_imputed = snp.mean_impute(df_filtered)
print(f"フィルタ後の形状: {df_filtered.shape}")
print(f"補完後の欠損数: {df_imputed.isna().sum().sum()}")

# ============================================================
# 4. メタデータ読み込み・並び替え
# ============================================================
print("\n---- 4. メタデータ読み込み・並び替え ----")
df_meta = snp.load_metadata(META_PATH)
print(f"メタデータ形状: {df_meta.shape}")

df_imputed = snp.reorder_by_metadata(df_imputed, df_meta)
df_not_imputed = snp.reorder_by_metadata(df_not_imputed, df_meta)
print(f"並び替え後の形状: {df_imputed.shape}")

---- 2. 欠損処理 ----


欠損処理: 欠損列除外=1318, 欠損行除外=0, 定数列除外=12475 (閾値=5%)


処理後の形状: (1476, 31002)

---- 3. 数値変換・MAF/HWE フィルタ・補完 ----


MAFフィルタ: 5140列除外 (MAF < 0.01)
HWEフィルタ: 475列除外 (p < 0.0001)


フィルタ後の形状: (1476, 25387)
補完後の欠損数: 0

---- 4. メタデータ読み込み・並び替え ----
メタデータ形状: (65008, 40)
並び替え後の形状: (1476, 25387)


## 4.5 特例補正（このデータ固有の既知の例外）

保存前に、特例の処理

1. **AX004**: genotyping TXT のCEL名が `AX004,AX007_...`（PJ番号を併記）。→ `AX004_...` に統一
2. **AX003_A13_2.CEL**: A13ウェルのリラン(取り直し)CEL。枝番 `_2` を落として `AX003_A13.CEL` に

In [11]:
# --- 何が変わるかを先に洗い出して可視化 ---
changes = [(s, fix_snp_id(s)) for s in df_imputed.index if fix_snp_id(s) != s]
print(f"特例補正: {len(changes)}件")
for old, new in changes:
    print(f"  {old}  ->  {new}")

# --- 補完版・非補完版の両方に適用 ---
for _d in (df_imputed, df_not_imputed):
    _d.index = [fix_snp_id(s) for s in _d.index]
    _d.index.name = "snp_id"

# 補正でsnp_idが衝突していないか確認
assert df_imputed.index.is_unique, "特例補正後に snp_id が重複しました"
assert df_not_imputed.index.is_unique, "特例補正後に snp_id が重複しました"

特例補正: 49件
  AX003_A13_2.CEL  ->  AX003_A13.CEL
  AX004,AX007_A01.CEL  ->  AX004_A01.CEL
  AX004,AX007_A03.CEL  ->  AX004_A03.CEL
  AX004,AX007_A05.CEL  ->  AX004_A05.CEL
  AX004,AX007_A07.CEL  ->  AX004_A07.CEL
  AX004,AX007_A09.CEL  ->  AX004_A09.CEL
  AX004,AX007_A11.CEL  ->  AX004_A11.CEL
  AX004,AX007_A13.CEL  ->  AX004_A13.CEL
  AX004,AX007_A15.CEL  ->  AX004_A15.CEL
  AX004,AX007_A17.CEL  ->  AX004_A17.CEL
  AX004,AX007_A19.CEL  ->  AX004_A19.CEL
  AX004,AX007_A21.CEL  ->  AX004_A21.CEL
  AX004,AX007_A23.CEL  ->  AX004_A23.CEL
  AX004,AX007_C01.CEL  ->  AX004_C01.CEL
  AX004,AX007_C03.CEL  ->  AX004_C03.CEL
  AX004,AX007_C05.CEL  ->  AX004_C05.CEL
  AX004,AX007_C07.CEL  ->  AX004_C07.CEL
  AX004,AX007_C09.CEL  ->  AX004_C09.CEL
  AX004,AX007_C11.CEL  ->  AX004_C11.CEL
  AX004,AX007_C13.CEL  ->  AX004_C13.CEL
  AX004,AX007_C15.CEL  ->  AX004_C15.CEL
  AX004,AX007_C17.CEL  ->  AX004_C17.CEL
  AX004,AX007_C19.CEL  ->  AX004_C19.CEL
  AX004,AX007_C21.CEL  ->  AX004_C21.CEL
  AX004,AX

## 5. 保存

`processed/snp_<YYYYMMDD>/` フォルダを作り、その中に保存する:

- `snp_imputed.feather` … 補完版
- `snp_not_imputed.feather` … 非補完版
- `manifest.json` … 入力TXT一覧・件数・閾値など（再現用）

In [12]:
# 保存先: processed/snp_<VERSION>/  （VERSION は手入力。バンプすると新フォルダに出る）
#   サブセット時は末尾にタグを付けて全件実行と分ける
VERSION = "20260708"
tag = "_" + "-".join(SELECT) if SELECT else ""
run_dir = PROCESSED_DIR / f"snp_{VERSION}{tag}"
run_dir.mkdir(parents=True, exist_ok=True)

out_imputed = run_dir / "snp_imputed.feather"
out_not_imputed = run_dir / "snp_not_imputed.feather"
out_manifest = run_dir / "manifest.json"

# snp_idを列に戻してから保存
df_imputed.reset_index().to_feather(out_imputed)
df_not_imputed.reset_index().to_feather(out_not_imputed)

# 再現用JSON
manifest = {
    "version": VERSION,
    "select": SELECT,
    "n_txt": len(txt_paths),
    "txt_files": [str(p) for p in txt_paths],
    "n_samples": int(len(df_imputed)),
    "n_snps": int(df_imputed.shape[1]),
    "thresholds": {
        "missing": MISSING_THRESHOLD,
        "maf": MAF_THRESHOLD,
        "hwe": HWE_THRESHOLD,
    },
}
out_manifest.write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)

print(f"保存先: {run_dir}")
print(f"  {out_imputed.name}")
print(f"  {out_not_imputed.name}")
print(f"  {out_manifest.name}")
print(f"サンプル数: {len(df_imputed)}, SNP数: {df_imputed.shape[1]}")

保存先: /home/s_mori/JBRT/JBRT_data/processed/snp_20260708
  snp_imputed.feather
  snp_not_imputed.feather
  manifest.json
サンプル数: 1476, SNP数: 25387


## 6. 前処理なしの素の行列を保存

フィルタ（欠損・定数列・MAF・HWE）・数値変換・平均補完は**どの個体集合で計算するかで結果が変わる**（コホート依存）。
ここで 1476 CEL（重複個体込み）に対して焼き付けてしまうと、下流で個体を絞ったときに合わなくなる。
そこで **1. の結合結果をそのまま**保存し、フィルタ類は実験側で各コホートに `jbrt.snp` の関数を適用して掛ける。

ここで触るのは **4.5 の snp_id 特例補正だけ**（IDの名寄せであってフィルタではない）。

- `snp_raw.feather` … AA/AB/BB の生値（無効値もそのまま）。1〜5 の出力とは独立。

In [13]:
# 1. の結合結果を作り直す（df は 2. 以降で上書きされているため dfs から取り直す）
df_raw = snp.concat_common_snps(dfs)

# snp_id の特例補正のみ適用（4.5 と同じ。フィルタは一切かけない）
df_raw.index = [fix_snp_id(s) for s in df_raw.index]
df_raw.index.name = "snp_id"
assert df_raw.index.is_unique, "特例補正後に snp_id が重複しました"

out_raw = run_dir / "snp_raw.feather"
df_raw.reset_index().to_feather(out_raw)

out_manifest_raw = run_dir / "manifest_raw.json"
out_manifest_raw.write_text(
    json.dumps(
        {
            "version": VERSION,
            "select": SELECT,
            "n_txt": len(txt_paths),
            "txt_files": [str(p) for p in txt_paths],
            "n_samples": int(len(df_raw)),
            "n_snps": int(df_raw.shape[1]),
            "filters": None,  # フィルタ・数値変換・補完は未適用（実験側で掛ける）
            "note": "共通SNPで結合 + snp_id特例補正のみ。生のAA/AB/BB（無効値も残置）。",
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print(f"保存先: {run_dir}")
print(f"  {out_raw.name}  ({out_raw.stat().st_size / 1e6:.1f} MB)")
print(f"  {out_manifest_raw.name}")
print(f"サンプル数: {len(df_raw)}, SNP数: {df_raw.shape[1]}")
df_raw.iloc[:3, :5]

保存先: /home/s_mori/JBRT/JBRT_data/processed/snp_20260708
  snp_raw.feather  (318.6 MB)
  manifest_raw.json
サンプル数: 1476, SNP数: 44795


probeset_id,AX-105094887,AX-106719429,AX-106719435,AX-106719437,AX-106719440
snp_id,,,,,
AX003_A01.CEL,AA,BB,BB,BB,AA
AX003_A03.CEL,AA,BB,BB,BB,AA
AX003_A05.CEL,AA,BB,BB,BB,AA
